#### Load Dataset

In [ ]:
from datasets import load_dataset
import os
import re
import numpy as np
import soundfile as sf
from collections import defaultdict

# Load TEDLIUM dataset, release 1
tedlium_dataset = load_dataset(
    "LIUM/tedlium",
    name="release1",
    split="test",
    streaming=True
)

#### Get n samples

In [ ]:
# Used to clean the transcript text
def clean_transcript(text):
    if "ignore_time_segment_in_scoring" in text:
        return ""
    return re.sub(r"\b(\w)\s+'(\w)", r"\1\2", text).strip()

def save_n_speakers(dataset, n=10, duration_sec=120, out_folder="data"):
    # Ensure output directory exists
    os.makedirs(out_folder, exist_ok=True)
    speaker_data = defaultdict(lambda: {"audio": [], "text": [], "sr": None})
    
    # Collect audio and text data for each speaker
    for sample in dataset:
        speaker = sample.get("speaker_id")
        if not speaker:
            continue
        text = clean_transcript(sample["text"])
        if not text:
            continue
        audio = sample["audio"]["array"]
        sr = sample["audio"]["sampling_rate"]
        speaker_data[speaker]["audio"].append(audio)
        speaker_data[speaker]["text"].append(text)
        speaker_data[speaker]["sr"] = sr

    # For each speaker, collect for a desired duration
    count = 0
    for speaker, data in speaker_data.items():
        sr = data["sr"]
        needed_samples = duration_sec * sr
        collected_audio, collected_text, collected = [], [], 0

        for audio, text in zip(data["audio"], data["text"]):
            if collected >= needed_samples:
                break
            take = min(len(audio), needed_samples - collected)
            collected_audio.append(audio[:take])
            frac = take / len(audio)
            if frac > 0.5:
                words = text.split()
                collected_text.append(" ".join(words[:int(len(words) * frac)]))
            collected += take

        # Skip if not enough audio collected for this speaker
        if collected < needed_samples:
            continue
        
        # Concatenate all collected audio and text
        out_audio = np.concatenate(collected_audio)
        out_text = " ".join(collected_text)
        count += 1
        # Save audio and text to files
        sf.write(os.path.join(out_folder, f"ted_sample_{count}_speaker_{speaker}.wav"), out_audio, sr)
        with open(os.path.join(out_folder, f"ted_sample_{count}_speaker_{speaker}.txt"), "w", encoding="utf-8") as f:
            f.write(out_text)
        print(f"Saved sample {count} for speaker {speaker}")

        if count >= n:
            break

save_n_speakers(tedlium_dataset, n=10, duration_sec=120, out_folder="data")